

In this article, I will try to solve the full MNIST challenge as part of the "Further research" section at the end of chapter 4 of the fastai book. This article was inspired by [Jack Driscoll's](https://medium.com/@jackdriscoll_90855/fast-ai-chapter-4-full-mnist-challenge-8558b1e0564a) implementation of MNIST using fastai and cross entropy loss. While none of the following code is new, I will try to explain cross entropy loss function and a similar one called binary cross entropy, how they're both equivalent, and also why it is preferred over other loss functions such as mse_loss (mean squared error loss). We will also going through the code for the challenge, using both fastai's learner and our own implementation of it. Anyone who has worked through chapter 4 of the fastai book should also be able to follow this. I will also cite a few extra resources which have helped me in writing this post. Let's get started!


First we need to initialise and setup fastai and pytorch libraries.

In [ ]:
#hide
! [ -e /content ] && pip install -Uqq fastbook
import fastbook
fastbook.setup_book()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.8/719.8 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.1/507.1 kB 16.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 28.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 10.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 11.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 35.1 MB/s eta 0:00:00
Mounted at /content/gdrive


In [ ]:
#hide
from fastai.vision.all import *
from fastbook import *

matplotlib.rc('image', cmap='Greys')

Now we need to install the MNIST dataset.

In [ ]:
path = untar_data(URLs.MNIST)
Path.BASE_PATH = path

Now we need to prepare our training and validation datasets from the MNIST dataset, which is structured as follows: The "path" variable in the above paragraph contains two folders, "training" and "testing".

In [ ]:
import numpy as np
train_path = (path/'training').ls().sorted() # a list containing paths of all digit folders
tensors = [(tensor(Image.open(o)), int(path.name)) for path in train_path for o in path.ls()] # list containing tuples for each input and its label in training dataset
np.random.shuffle(tensors)


In [ ]:
index = math.floor(len(tensors) * 0.8)
train, valid = tensors[:index], tensors[index:] # splitting "tensors" into training and validation tensors
train_x = torch.cat([train[i][0].view(-1, 784).float()/255 for i in range(len(train))]) # converting each tensor from [28,28] to [1,784] shape
                                                                                          # for matrix multiplication
train_y = tensor([train[i][1] for i in range(len(train))]) # separating out xs and ys into separate tensors
dset = list(zip(train_x, train_y))
valid_x = torch.cat([valid[i][0].view(-1, 784).float()/255 for i in range(len(valid))])
valid_y = tensor([valid[i][1] for i in range(len(valid))])
valid_dset = list(zip(valid_x, valid_y))

Now we create our training and validation dataloaders, with a mini-batch size of 256.

In [ ]:
dl = DataLoader(dset, batch_size=256)
valid_dl = DataLoader(valid_dset, batch_size = 256)
dls = DataLoaders(dl, valid_dl)

##Introducing Cross entropy loss function

Cross entropy is a loss function which can be used as a replacement for the mean squared error loss (mse loss) used in chapter 4 of the book. Here's the expression for what is called the "binary cross-entropy loss function"(this is used when y is either 0 or 1).

\begin{eqnarray}
  L = -\frac{1}{n} \sum_x \left[y \ln a + (1-y ) \ln (1-a) \right],\tag{1}
\end{eqnarray}
where n
 is the total number of items of training data, the sum is over all training inputs, x
, y
 is the corresponding desired output, and a is the output activation.
To see why this even makes sense as a loss function, consider that any loss function satisfies two properties:

1.   ${L}$ is always non-negative( ${L} \ge 0 $)
2.   Smaller the absolute difference between y and $a$, smaller the value of ${L}$ must turn out to be.

To see how ${L}$ satisfies property 1, consider how the the value of a is always between 0 and 1, since it is passed through a function such as [sigmoid](https://en.wikipedia.org/wiki/Sigmoid_function) before it enters the expression for ${L}$. Hence the $\ln(a)$ is always negative, and since  there is already a $-$ sign in the expression, the overall value of ${L}$ is always non-negative.

One can also verify property 2 easily. Lets say y = 1 (for an individual training input). Then the expression for ${L}$ reduces to $ L = - \ln a $. Now the closer the value of a is to 1, the closer ${L}$ will get to 0. Conversely, the closer a is to the other end, i.e., 0, the larger ${L}$ will turn out to be. Similarly, one can see the same pattern for y = 0; now ${L}$ reduces to $ L = -y\ln (1-a) $. Here ${L}$ gets close to 0 as a gets close to 0, and ${L}$ increases exponentially as a increases

Now that we have seen binary cross-entropy loss function (y can take only one of 2 values), let's see how we can extend this concept to $K$ classes (y can take any one of K classes), in which the expression for ${L}$ takes a slightly different form, albeit equivalent to the binary cross-entropy function(I will explain how later):


$L=\frac{1}{n}\sum_{i=1}^{n} \sum_{k=1}^{K}-y_{k}^{[i]} \ln \left(a_{k}^{[i]}\right)\tag{2}.$

Here, both y is assumed to be one-hot encoded. For example, if y = 5, it must be converted to its one-hot encoded form [0, 0, 0, 0, 0, 1, 0, 0, 0, 0]. So the *index* of the new one-hot encoded y represents the actual label. And also, the output activations from the model (the 10D vector) are passed into the [softmax](https://en.wikipedia.org/wiki/Softmax_function) function, which converts the output activations into a probability distribution. So the values of $a$ are the softmax'd values of the actual output activations from the model.

Since $y_{k} = 0$ (in its one-hot encoded form) for all k except the actual label (the number, eg. 4), say $Y$, we are essentially calculating $L=\frac{1}{n}\sum_{i=1}^{n} -y_{Y}^{[i]} \ln \left(a_{Y}^{[i]}\right)$. And since $y_{Y} = 1$, we have:
 $L=\frac{1}{n}\sum_{i=1}^{n} -\ln \left(a_{Y}^{[i]}\right)\tag{3}$.

 For example, let's say $y$ = [0, 0, 1, 0, 0, 0, 0, 0, 0, 0], and $a$ = [0.1, 0, 0.8, 0, 0, 0.05, 0, 0.05, 0, 0]. Since $Y$ = 2, we calculate loss at **index 2**, i,e., $L = -\ln(0.8)$.

Hence closer the value of $a_{Y}^{[i]}$ is to $y_{Y}$ (which is always 1), smaller the value of L is going to be.

At this point, you may be wondering: Fine, but what is the aforementioned equivalence between binary cross-entropy and the generalised cross-entropy?

To gain insight into this, observe that the binary cross entropy is calculated  for two scalars $a$ and $y$. Here we are trying to predict the labels themselves (0 or 1). If a label $y$ is 1, we want our output $a$ to be close to 1. Hence we apply the loss function $ L = - \ln a $. If the label $y$ is 0 instead, then we want our output $a$ to be close to 0. This is where the other part $(1-y ) \ln (1-a)$ in $(1)$comes in. It calcualates the loss when the label $y$ is a 0. Whereas this part wouldn't matter in the one-hot encoded forms in multi-categorical case (general cross-entropy function), since in this case the actual label $Y$ doesn't matter anymore, it is just converted to a 1 at index no. $Y$, rest all 0s. And we want our output $a$ (in this case, it's a 10D vector) to be 1 only at that index $Y$, what values it spits out at other indices don't matter at all. We calculate the loss $-\ln a_{Y}$, ignoring the other indices. Note that there is no $(1-y ) \ln (1-a)$ part required since the loss function only ever calculates loss at the index which is a 1, eliminating the necessity to calculate loss when the label is a 0.


So while binary cross-entropy and the generalised form of cross-entropy are essentially calculating the same thing, the former also accounts for an extra case, i.e., when the label $y = 0$.

For more information on this, I'd recommend [this article](https://sebastianraschka.com/faq/docs/pytorch-crossentropy.html) by Sebastian Raschka, which also delves into the actual functions in pytorch used to implement these loss functions.

Also, to understand more about why cross entropy is used as a replacement for MSE loss, I'd recommend this [intuitive explantion](http://neuralnetworksanddeeplearning.com/chap3.html#the_cross-entropy_cost_function) from Michael Nielsen's book. Note that you need to understand backpropagation to understand the above, and I'd recommend [chapter 2](http://neuralnetworksanddeeplearning.com/chap2.html) from the same book for it.

We implement what we have learned above using Pytorch's torch.nn.functional.cross_entropy function. Note that we directly pass in the output predictions, without softmax'ing them, because the cross_entropy function does it for us.  Also note that we don't have to pass in one-hot encoded labels, instead we can directly pass in the actual labels themselves, i.e., the numbers. This is because in pytorch, $L$ is not actually calculated as per equation $(1)$, but more like equation $(3)$. [Source](https://stackoverflow.com/questions/62456558/is-one-hot-encoding-required-for-using-pytorchs-cross-entropy-loss-function#:~:text=CrossEntropyLoss%20expects%20integer%20labels.,class%20as%20the%20final%20label.)

In [ ]:
def mnist_loss(preds, targets):

  loss = torch.nn.functional.cross_entropy(preds, targets)
  return loss


Now we create a simple neural network:

In [ ]:
simple_net = nn.Sequential(
    nn.Linear(28*28,30),
    nn.ReLU(),
    nn.Linear(30,10)
)

We now create a function to calculate the accuracy of a mini-batch. Here we hace to softmax the outputs, unlike in the mnist_loss function where the cross_entropy function did it for us. torch.argmax retrives the index of a tensor with the highest value, which is going to be the value our model predicts.

In [ ]:
def batch_accuracy(xb, yb):
  preds = F.softmax(xb, dim=1) # dimension 0(rows) contains number of xs in xb(mini-batch), wheareas dimension 1 contains the actual values in each x
  corrects = torch.argmax(preds, dim=1) == yb
  return corrects.float().mean()

In [ ]:
learn = Learner(dls, simple_net, opt_func=SGD, loss_func=mnist_loss, metrics=batch_accuracy)


In [ ]:
learn.fit(20, lr=1.)

/usr/local/lib/python3.10/dist-packages/fastai/torch_core.py:263: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  return getattr(torch, 'has_mps', False)


epoch,train_loss,valid_loss,batch_accuracy,time
0,0.493051,0.413100,0.872167,00:01
1,0.312087,0.327860,0.902250,00:01
2,0.255675,0.300931,0.908333,00:01
3,0.231685,0.275073,0.918667,00:01
4,0.215447,0.257873,0.922833,00:01
5,0.198391,0.248927,0.925500,00:00
6,0.181042,0.235061,0.930667,00:00
7,0.168435,0.223448,0.935000,00:00
8,0.159829,0.218092,0.935917,00:00
9,0.152646,0.220556,0.936500,00:00


93.2% accuracy in the first run! While we could do much better using better architectures like CNNs etc, I am gonna stop here as this article was intented to be more of an explainer of the cross-entropy function, and how it can be applied to a practical dataset such as MNIST. Thanks for reading!